# Disambiguation of Abbreviations and Acronyms for NLP Applications

Code accompanying the paper:

> N. Rizk, S. A. Alansary, A. M. R. M. Seddeek, N. Sarwat. *Disambiguation of Abbreviations and Acronyms for Natural Language Processing Applications*. IEEE, 2026. DOI: [10.1109/LT68265.2026.11592568](https://doi.org/10.1109/LT68265.2026.11592568)

A single abbreviation or acronym (A&A) such as **AAA** can stand for many expansions (*American Association for Anatomy*, *Abdominal Aortic Aneurysm*, *Aromatic Amino Acid*, ...). This notebook detects the A&As in a text and selects the expansion that best fits the context.

**Pipeline**

1. **Pre-processing**: normalise and tokenise the text.
2. **Detection**: find candidate A&As with rule-based patterns.
3. **Text classification**: assign a domain to the text with zero-shot BART (`facebook/bart-large-mnli`).
4. **In-text expansion**: if the expansion is written in the text, find it by matching the A&A letters with the initials of n-grams.
5. **External candidates**: otherwise, retrieve candidate expansions from the Abbreviations.com (STANDS4) API.
6. **Linguistic scoring**: POS-tag the candidates and score each one by how often its content words occur in the text; the highest score wins.

**Results** (279 A&As, 279 multi-domain Wikipedia texts, 71,746 words): 260 / 279 correct, **accuracy 93.19%**.

## 1. Setup

The external API needs a free STANDS4 account (https://www.abbreviations.com/api.php). Store your credentials as **Colab secrets** named `STANDS4_UID` and `STANDS4_TOKEN` (key icon in the left sidebar), or as environment variables when running locally. Never write them directly in the notebook.

In [ ]:
!pip install -q transformers nltk requests

In [ ]:
import os
import re

import nltk
import requests
from nltk import pos_tag, word_tokenize
from nltk.tokenize import TreebankWordTokenizer
from nltk.util import everygrams
from transformers import pipeline

for resource in ["punkt", "punkt_tab", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(resource, quiet=True)

In [ ]:
def get_secret(name):
    """Read a credential from Colab secrets, falling back to environment variables."""
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)


STANDS4_UID = get_secret("STANDS4_UID")
STANDS4_TOKEN = get_secret("STANDS4_TOKEN")

## 2. Text classification (zero-shot BART)

In [ ]:
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

DOMAIN_LABELS = [
    "Academic & Science", "Business", "Community", "Computing", "Governmental",
    "Internet", "Miscellaneous", "Regional", "Medical", "International",
]


def classify_text(text):
    """Return the most likely domain of the text."""
    prediction = classifier(
        text,
        DOMAIN_LABELS,
        hypothesis_template="This text is about {}.",
        multi_label=True,
    )
    return prediction["labels"][0]

## 3. Pre-processing

In [ ]:
_tokenizer = TreebankWordTokenizer()


def clean_text(text):
    """Normalise apostrophes and drop characters outside letters, digits and basic punctuation."""
    text = text.replace("\u2019", "'")
    return re.sub(r"[^a-zA-Z0-9\s.()',\-]", "", text)


def tokenize(text):
    return _tokenizer.tokenize(text)

## 4. Abbreviation detection

Rule-based patterns for upper-case tokens (`NLP`), plurals (`CEOs`), hyphenated forms (`COVID-19`, `non-EU`) and dotted forms (`U.S.A.`). The value stored for each A&A is the number of letters, used later to pick n-grams of the same length.

In [ ]:
def detect_abbs(tokens):
    abbs = {}
    for i, token in enumerate(tokens):
        if (0 < i < len(tokens) - 1
                and token.isupper() and token.isalpha() and len(token) >= 2
                and not tokens[i - 1].isalpha() and not tokens[i + 1].isupper()):
            abbs[token] = len(token)
        elif token[:-1].isupper() and token[-1] == "s" and len(token) >= 3:
            abbs[token[:-1]] = len(token) - 1
        elif "-" in token:
            dash = token.index("-")
            left, right = token[:dash], token[dash + 1:]
            if left.isupper() and right.isupper():
                abbs[token] = len(token) - token.count("-")
            elif left.isupper() and right.isdigit():
                abbs[token] = len(left)
            elif not left.isupper() and right.isupper():
                abbs[right] = len(right)
        elif "." in token:
            if re.fullmatch(r"(\A[A-Z]\.)([A-Z]\.)+([A-Z]\Z)?(\.)?", token):
                abbs[token.replace(".", "")] = len(token) - token.count(".")
        elif token.isupper() and len(token) >= 2:
            abbs[token] = len(token)
    return abbs

## 5. In-text expansion (n-gram matching)

If the full form appears in the text (for example *Natural Language Processing (NLP)*), it is found by comparing the A&A letters with the initials of n-grams of the same length.

In [ ]:
def get_ngrams(tokens, max_len):
    grams = {n: [] for n in range(2, max_len + 1)}
    for gram in everygrams(tokens, 2, max_len):
        grams[len(gram)].append(gram)
    return grams


def search_in_text(abbs, ngrams):
    results = []
    for abb, length in abbs.items():
        letters = re.sub(r"[^A-Za-z]", "", abb)
        matches = [
            gram for gram in ngrams.get(length, [])
            if "".join(w[0] for w in gram if w[0].isupper()) == letters
        ]
        results.append((abb, matches))
    return results

## 6. External candidates (Abbreviations.com API)

In [ ]:
API_URL = "https://www.stands4.com/services/v2/abbr.php"


def get_candidates(abb):
    """Return the list of candidate expansions for an abbreviation."""
    if not STANDS4_UID or not STANDS4_TOKEN:
        raise RuntimeError("Set STANDS4_UID and STANDS4_TOKEN first (see Setup).")
    response = requests.get(
        API_URL,
        params={"uid": STANDS4_UID, "tokenid": STANDS4_TOKEN, "term": abb, "format": "json"},
        headers={"User-Agent": "colab-notebook"},
        timeout=30,
    )
    response.raise_for_status()
    result = response.json().get("result", [])
    if isinstance(result, dict):  # a single result comes back as an object
        result = [result]
    return sorted({item["definition"] for item in result})

## 7. Linguistic scoring

Each candidate is POS-tagged. Its content words are matched against the content words of the text, and each match adds the word's frequency in the text to the candidate's score. The candidate with the highest score is selected.

In [ ]:
CONTENT_TAG_PREFIXES = ("N", "V", "J", "R")  # nouns, verbs, adjectives, adverbs


def content_words(text):
    return [word.lower() for word, tag in pos_tag(word_tokenize(text))
            if tag.startswith(CONTENT_TAG_PREFIXES)]


def score_candidates(text, candidates):
    """Return the best candidate and the score of every candidate."""
    text_words = content_words(text)
    scores = {
        candidate: sum(text_words.count(word) for word in content_words(candidate))
        for candidate in candidates
    }
    return max(scores, key=scores.get), scores

## 8. Full pipeline

In [ ]:
def disambiguate(text, verbose=True):
    """Detect the A&As in a text and return the selected expansion for each one."""
    cleaned = clean_text(text)
    tokens = tokenize(cleaned)
    category = classify_text(text)
    abbs = detect_abbs(tokens)
    if not abbs:
        if verbose:
            print("No abbreviations found in the text.")
        return {"category": category, "results": []}

    ngrams = get_ngrams(tokens, max(abbs.values()))
    results = []
    for abb, matches in search_in_text(abbs, ngrams):
        if matches:
            results.append({"abbreviation": abb, "expansion": " ".join(matches[0]), "source": "text"})
            continue
        try:
            candidates = get_candidates(abb)
        except Exception as error:
            results.append({"abbreviation": abb, "expansion": None, "source": f"error: {error}"})
            continue
        if not candidates:
            results.append({"abbreviation": abb, "expansion": None, "source": "not found"})
            continue
        best, _ = score_candidates(cleaned, candidates)
        results.append({"abbreviation": abb, "expansion": best, "source": "api + scoring"})

    if verbose:
        print("Text category:", category)
        for r in results:
            print(f"  {r['abbreviation']:<10} -> {r['expansion']}  ({r['source']})")
    return {"category": category, "results": results}

## 9. Example

In [ ]:
example = (
    "The Kenya Air Force (KAF) is the national aerial warfare service branch of the "
    "Republic of Kenya. The Kenya Air Force flies some two dozen fighter jets, "
    "a dozen Tucano trainers and several dozen helicopters."
)
disambiguate(example)

In [ ]:
# Try your own text
# disambiguate("Paste a paragraph containing abbreviations here.")